In [1]:
from pathlib import Path
import numpy as np
import os
import pandas as pd
import re
import time

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')
print(sys.path)

['/home/hieutt/CAN-SupCon-IDS/notebooks', '/home/hieutt/miniconda3/envs/torchtf/lib/python39.zip', '/home/hieutt/miniconda3/envs/torchtf/lib/python3.9', '/home/hieutt/miniconda3/envs/torchtf/lib/python3.9/lib-dynload', '', '/home/hieutt/miniconda3/envs/torchtf/lib/python3.9/site-packages', '../']


In [2]:
import os
import re
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- CONFIG ---
input_dir = '../data/2017-subaru-forester/post-attack-labeled'
output_dir = '../data/2017-subaru-forester/merged'
os.makedirs(output_dir, exist_ok=True)

attack_types = [
    'combined-attacks', 'DoS-attacks', 'fuzzing-attacks', 'gear-attacks',
    'interval-attacks', 'rpm-attacks', 'speed-attacks', 'standstill-attacks', 'systematic-attacks'
]

USECOLS = ['timestamp', 'arbitration_id', 'data_field', 'attack']
HEX_RE = re.compile(r'^[0-9a-fA-F]+$')

def parse_arbitration_id(v):
    """
    Robust parse arbitration_id that may be:
    - decimal int (e.g., 356)
    - hex string (e.g., '0x164', '164', '1A3')
    """
    if pd.isna(v):
        return -1
    s = str(v).strip().lower()
    if s == '':
        return -1

    # remove spaces
    s = s.replace(' ', '')

    # detect hex
    is_hex = False
    if s.startswith('0x'):
        is_hex = True
        s = s[2:]
    else:
        # contains a-f => hex
        if any(ch in 'abcdef' for ch in s):
            is_hex = True

    # if still purely digits and you want to treat as decimal -> keep decimal
    # (common in already-cleaned CAN-ML exports)
    base = 16 if is_hex else 10

    try:
        return int(s, base)
    except:
        return -1

def normalize_payload_hex(v):
    """
    Keep payload in HEX (variable length allowed).
    - remove spaces
    - lower-case
    - drop 0x prefix
    - keep only hex chars
    - if odd length -> pad a leading '0' to make byte-pairs valid
    """
    if pd.isna(v):
        return ''
    s = str(v).strip().lower()
    if s == '':
        return ''

    s = s.replace(' ', '')
    if s.startswith('0x'):
        s = s[2:]

    # keep only hex chars
    s = ''.join(ch for ch in s if ch in '0123456789abcdef')
    if s == '':
        return ''

    # make even-length
    if len(s) % 2 == 1:
        s = '0' + s

    return s

for attack_type in attack_types:
    attack_folder = os.path.join(input_dir, attack_type)
    if not os.path.isdir(attack_folder):
        continue

    print(f"\nProcessing group: {attack_type} ...")
    all_files = [os.path.join(attack_folder, f) for f in os.listdir(attack_folder) if f.endswith('.csv')]
    if not all_files:
        continue

    df_list = []
    for file_path in tqdm(all_files, desc="Reading files"):
        try:
            df = pd.read_csv(file_path, dtype=str)  # read as str to avoid weird parsing
            # make sure required columns exist
            if not all(c in df.columns for c in USECOLS):
                continue

            # filter bad timestamp rows
            ts = pd.to_numeric(df['timestamp'], errors='coerce')
            df = df[ts.notna()].copy()
            df['timestamp'] = ts[df.index].astype(np.float64)

            # keep only needed cols
            df = df[USECOLS]
            df_list.append(df)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    if not df_list:
        continue

    full_df = pd.concat(df_list, ignore_index=True)

    # sort by time
    full_df = full_df.sort_values(by='timestamp').reset_index(drop=True)

    # parse arbitration_id
    full_df['arbitration_id'] = full_df['arbitration_id'].apply(parse_arbitration_id)
    full_df = full_df[full_df['arbitration_id'] != -1].copy()

    # normalize payload (HEX)
    full_df['data_field'] = full_df['data_field'].apply(normalize_payload_hex)

    # attack -> int 0/1
    full_df['attack'] = pd.to_numeric(full_df['attack'], errors='coerce').fillna(0).astype(np.int8)
    full_df['attack'] = (full_df['attack'] > 0).astype(np.int8)

    # final sanity: drop empty payload rows if you want
    # full_df = full_df[full_df['data_field'] != ''].copy()

    out_name = attack_type.replace('-attacks', '').lower() + '.csv'
    out_path = os.path.join(output_dir, out_name)

    full_df[['timestamp', 'arbitration_id', 'data_field', 'attack']].to_csv(out_path, index=False)
    print(f"Saved: {out_path} | rows={len(full_df)}")



Processing group: combined-attacks ...


Reading files: 100%|██████████| 8/8 [00:07<00:00,  1.07it/s]


Saved: ../data/2017-subaru-forester/merged/combined.csv | rows=5806825

Processing group: DoS-attacks ...


Reading files: 100%|██████████| 4/4 [00:02<00:00,  1.42it/s]


Saved: ../data/2017-subaru-forester/merged/dos.csv | rows=2297897

Processing group: fuzzing-attacks ...


...

Saved: ../data/2017-subaru-forester/merged/speed.csv | rows=3672497

Processing group: standstill-attacks ...


Reading files: 100%|██████████| 4/4 [00:02<00:00,  1.38it/s]


Saved: ../data/2017-subaru-forester/merged/standstill.csv | rows=2361472

Processing group: systematic-attacks ...


Reading files: 100%|██████████| 4/4 [00:04<00:00,  1.05s/it]


Saved: ../data/2017-subaru-forester/merged/systematic.csv | rows=3398646


In [3]:
import pandas as pd
df = pd.read_csv('../data/2017-subaru-forester/merged/dos.csv')
dt = df['timestamp'].diff().dropna()
print("unique dt (first 10):", dt.unique()[:10])
print("dt quantiles:", dt.quantile([0,0.5,0.9,0.99]).to_dict())


unique dt (first 10): [0.00000000e+00 2.71797180e-05 6.91413879e-06 5.00679016e-06
 1.11889839e-03 1.00135803e-05 3.21865082e-05 1.97887421e-05
 8.10623169e-06 5.96046448e-06]
dt quantiles: {0.0: 0.0, 0.5: 1.2159347534179688e-05, 0.9: 0.0010900497436523438, 0.99: 0.004242897033691406}


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/2017-subaru-forester/merged/dos.csv')
t = df['timestamp'].to_numpy()

T = 0.01  # 10ms
stride = 0.01
counts = []
start = t[0]
end = t[-1]

cur = start
j0 = 0
n = len(t)
while cur + T <= end:
    # advance left
    while j0 < n and t[j0] < cur:
        j0 += 1
    j1 = j0
    while j1 < n and t[j1] < cur + T:
        j1 += 1
    counts.append(j1 - j0)
    cur += stride

counts = np.array(counts)
print("count quantiles:", np.quantile(counts, [0,0.5,0.9,0.99]))


count quantiles: [ 0. 21. 49. 65.]


In [5]:
import pandas as pd
import numpy as np

fp = "../data/2017-subaru-forester/merged/dos.csv"
df = pd.read_csv(fp, usecols=["timestamp","attack"])
df["timestamp"] = df["timestamp"].astype(float)
df = df.sort_values("timestamp")

atk = df["attack"].to_numpy()
n = len(atk)

# chia file thành 10 bins theo thời gian
bins = np.array_split(atk, 10)
print([float(b.mean()) for b in bins])  # tỷ lệ attack trong mỗi 10% thời gian


[0.03033639409895992, 0.02085817485530267, 0.0249053483615475, 0.0526176073806519, 0.01852996213934462, 0.035388833282562336, 0.0, 0.04277837494397034, 0.0, 0.0]
